# <font color="#418FDE" size="6.5" uppercase>**Gaussian Processes**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Fit Gaussian Process Regression and interpret predictive mean and uncertainty. 
- Configure and compose Gaussian process kernels for practical modeling. 
- Apply Gaussian Process Classification while recognizing computational limits. 


## **1. GPR Basics**

### **1.1. GP Distributions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_01_01.jpg?v=1784003024" width="250">



>* Models many possible functions, not one curve
>* Predicts expected values with uncertainty ranges

>* Function values are modeled jointly as Gaussian
>* Kernels link nearby inputs and uncertainty

>* Predict mean estimates expected function values
>* Uncertainty grows farther from observed data



### **1.2. Fitting GPR Models**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_01_02.jpg?v=1784003026" width="250">



>* Training data updates likely functions
>* Posterior predicts values and uncertainty

>* Predictive mean estimates outputs from learned patterns
>* Sparse data makes assumptions more influential

>* Uncertainty shows confidence in each prediction
>* Higher uncertainty marks less supported predictions



In [ ]:
#@title Python Code - Fitting GPR Models

# Fit a Gaussian Process Regression model.
# Show predictive mean and uncertainty.
# Visualize confidence near observed data.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.gaussian_process.kernels import WhiteKernel

# Create a small one-dimensional regression dataset.
rng = np.random.default_rng(42)
X_train = np.linspace(0, 10, 14).reshape(-1, 1)
y_train = np.sin(X_train).ravel() + rng.normal(0, 0.15, X_train.shape[0])

# Validate the basic training shapes before fitting.
if X_train.shape[0] != y_train.shape[0]:
    raise ValueError("Each input must have one target value.")

# Combine a smoothness kernel with a noise kernel.
kernel = RBF(length_scale=1.5) + WhiteKernel(noise_level=0.04)
model = GaussianProcessRegressor(kernel=kernel, random_state=42)

# Fit once, then predict means and standard deviations.
model.fit(X_train, y_train)
X_test = np.linspace(-1, 11, 250).reshape(-1, 1)
mean_prediction, std_prediction = model.predict(X_test, return_std=True)

# Print concise facts that support the plot interpretation.
print(f"scikit-learn version: {sklearn.__version__}")
print(f"Training points: {X_train.shape[0]}")
print(f"Mean uncertainty near data: {std_prediction[80:170].mean():.3f}")
print(f"Mean uncertainty at edges: {np.r_[std_prediction[:25], std_prediction[-25:]].mean():.3f}")

# Plot the fitted mean and its uncertainty band.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_train.ravel(), y_train, color="black", label="observed data")
ax.plot(X_test.ravel(), mean_prediction, color="tab:blue", label="predictive mean")

# The shaded band shows about two standard deviations.
ax.fill_between(
    X_test.ravel(),
    mean_prediction - 2 * std_prediction,
    mean_prediction + 2 * std_prediction,
    color="tab:blue",
    alpha=0.2,
    label="uncertainty band",
)

# Labels make the regression curve easier to interpret.
ax.set_title("Gaussian Process Regression fit")
ax.set_xlabel("Input value")
ax.set_ylabel("Predicted output")
ax.legend()
plt.show()



### **1.3. Alpha Noise**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_01_03.jpg?v=1784003028" width="250">



>* Alpha noise represents measurement error in observations
>* It smooths predictions and reflects data uncertainty

>* Low alpha closely fits training points
>* Higher alpha smooths isolated noisy deviations

>* Noise level shapes confidence bands
>* Choose alpha based on data reliability



In [ ]:
#@title Python Code - Alpha Noise

# Compare alpha noise in Gaussian process regression.
# Alpha controls trust in noisy training observations.
# The plot shows smoothing and uncertainty changes.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

# Create a small noisy one-dimensional regression dataset.
rng = np.random.default_rng(42)
X_train = np.linspace(0, 10, 18).reshape(-1, 1)
true_signal = np.sin(X_train).ravel()
y_train = true_signal + rng.normal(0, 0.35, size=X_train.shape[0])

# Validate the simple shape expected by the model.
if X_train.shape[0] != y_train.shape[0]:
    raise ValueError("Training inputs and targets must have matching rows.")

# Predict on a smooth grid for easier visual comparison.
X_grid = np.linspace(0, 10, 300).reshape(-1, 1)
kernel = 1.0 * RBF(length_scale=1.2)

# Fit one model that nearly trusts every observation.
gpr_low_alpha = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,
    optimizer=None,
    normalize_y=True,
)
gpr_low_alpha.fit(X_train, y_train)

# Fit one model that assumes visible measurement noise.
gpr_high_alpha = GaussianProcessRegressor(
    kernel=kernel,
    alpha=0.12,
    optimizer=None,
    normalize_y=True,
)
gpr_high_alpha.fit(X_train, y_train)

# Get predictive means and standard deviations.
mean_low, std_low = gpr_low_alpha.predict(X_grid, return_std=True)
mean_high, std_high = gpr_high_alpha.predict(X_grid, return_std=True)

# Print concise numeric clues before the visual comparison.
print(f"scikit-learn version: {sklearn.__version__}")
print(f"Average uncertainty, tiny alpha: {np.mean(std_low):.3f}")
print(f"Average uncertainty, larger alpha: {np.mean(std_high):.3f}")

# Plot both fitted means and one uncertainty band.
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(X_train.ravel(), y_train, color="black", label="noisy observations")
ax.plot(X_grid.ravel(), mean_low, color="tab:blue", label="alpha = 1e-6")
ax.plot(X_grid.ravel(), mean_high, color="tab:orange", label="alpha = 0.12")

# Show uncertainty for the larger-alpha model.
ax.fill_between(
    X_grid.ravel(),
    mean_high - 2 * std_high,
    mean_high + 2 * std_high,
    color="tab:orange",
    alpha=0.2,
    label="larger-alpha uncertainty band",
)

# Label the figure for beginner interpretation.
ax.set_title("Alpha noise lets GPR smooth noisy observations")
ax.set_xlabel("Input value")
ax.set_ylabel("Observed target")
ax.legend()
plt.show()



## **2. Kernel Design**

### **2.1. Core GP Kernels**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_02_01.jpg?v=1784003013" width="250">



>* Kernels encode prior assumptions about function behavior
>* They shape predictions and uncertainty estimates

>* Squared exponential kernels model very smooth changes
>* Matérn kernels handle rougher continuous patterns

>* Linear, periodic, and noise kernels model patterns
>* Match kernels to data assumptions and evidence



In [ ]:
#@title Python Code - Core GP Kernels

# Compare core Gaussian process kernels.
# Kernels encode smoothness, trend, cycles, and noise.
# The plot shows different prior assumptions.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.gaussian_process.kernels import RBF
from sklearn.gaussian_process.kernels import Matern
from sklearn.gaussian_process.kernels import DotProduct
from sklearn.gaussian_process.kernels import ExpSineSquared
from sklearn.gaussian_process.kernels import WhiteKernel

# Create one-dimensional input locations for kernel comparison.
x_values = np.linspace(0.0, 10.0, 200).reshape(-1, 1)
reference_point = np.array([[5.0]])

# Validate the simple shape expected by these kernels.
if x_values.shape != (200, 1):
    raise ValueError("Expected 200 one-dimensional input points.")

# Define core kernels with beginner-friendly parameter choices.
kernels = {
    "RBF smooth": RBF(length_scale=1.2),
    "Matern rougher": Matern(length_scale=1.2, nu=1.5),
    "Linear trend": DotProduct(sigma_0=1.0),
    "Periodic cycle": ExpSineSquared(length_scale=1.0, periodicity=3.0),
    "White noise": WhiteKernel(noise_level=1.0),
}

# Measure similarity between every point and the reference point.
similarities = {}
for kernel_name, kernel in kernels.items():
    similarities[kernel_name] = kernel(x_values, reference_point).ravel()

print("scikit-learn version:", sklearn.__version__)
print("Reference input value:", float(reference_point[0, 0]))
print("Higher curves mean stronger kernel similarity.")

# Plot each kernel's similarity pattern on one shared axis.
fig, ax = plt.subplots(figsize=(9, 5))
for kernel_name, similarity in similarities.items():
    ax.plot(x_values.ravel(), similarity, label=kernel_name)

ax.set_title("Core Gaussian Process Kernels Around One Reference Point")
ax.set_xlabel("Input value")
ax.set_ylabel("Kernel similarity to x = 5")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()



### **2.2. Combining Kernels**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_02_02.jpg?v=1784003015" width="250">



>* Combine kernels to capture multiple data patterns
>* Model predictions and uncertainty more flexibly

>* Addition combines independent pattern components
>* Multiplication models interactions between patterns

>* Start simple; add kernels with clear purpose
>* Balance flexibility, interpretability, and reliable predictions



In [ ]:
#@title Python Code - Combining Kernels

# This example combines Gaussian process kernels.
# Addition and multiplication encode different assumptions.
# The plot compares uncertainty from composed kernels.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel
from sklearn.gaussian_process.kernels import ExpSineSquared
from sklearn.gaussian_process.kernels import RBF
from sklearn.gaussian_process.kernels import WhiteKernel

# Synthetic data has a smooth trend and repeating wiggles.
rng = np.random.default_rng(42)
X_train = np.linspace(0, 10, 28).reshape(-1, 1)
trend = 0.35 * X_train.ravel()
seasonal = np.sin(2 * np.pi * X_train.ravel() / 3)
noise = rng.normal(0, 0.18, size=X_train.shape[0])
y_train = trend + seasonal + noise

# Validate the simple one-dimensional training data.
if X_train.shape[0] != y_train.shape[0]:
    raise ValueError("Training inputs and targets must match.")

# Addition says trend and seasonality coexist independently.
trend_kernel = ConstantKernel(1.0) * RBF(length_scale=4.0)
seasonal_kernel = ConstantKernel(1.0) * ExpSineSquared(
    length_scale=1.0,
    periodicity=3.0,
)
noise_kernel = WhiteKernel(noise_level=0.05)
combined_kernel = trend_kernel + seasonal_kernel + noise_kernel

# One Gaussian process learns the composed covariance pattern.
model = GaussianProcessRegressor(
    kernel=combined_kernel,
    alpha=0.0,
    normalize_y=True,
    random_state=42,
)
model.fit(X_train, y_train)

# Predictions include both a mean curve and uncertainty band.
X_test = np.linspace(0, 12, 240).reshape(-1, 1)
mean_prediction, std_prediction = model.predict(X_test, return_std=True)

print("scikit-learn version:", sklearn.__version__)
print("Kernel idea: smooth trend + periodic pattern + noise.")
print("Learned kernel:", model.kernel_)

# The shaded band shows uncertainty from the combined kernel.
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(X_train.ravel(), y_train, color="black", s=28, label="training data")
ax.plot(X_test.ravel(), mean_prediction, color="tab:blue", label="GP mean")
ax.fill_between(
    X_test.ravel(),
    mean_prediction - 2 * std_prediction,
    mean_prediction + 2 * std_prediction,
    color="tab:blue",
    alpha=0.18,
    label="about 95% uncertainty",
)
ax.set_title("Gaussian Process with Added Kernels")
ax.set_xlabel("Time")
ax.set_ylabel("Observed value")
ax.legend()
plt.show()



### **2.3. Tuning Kernel Parameters**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_02_03.jpg?v=1784003017" width="250">



>* Kernel parameters shape expected data patterns
>* They affect predictions, uncertainty, and generalization

>* Balance data fit with model complexity
>* Use domain judgment when tuning parameters

>* Start with assumptions, then evaluate predictions
>* Check combined kernels and uncertainty behavior



In [ ]:
#@title Python Code - Tuning Kernel Parameters

# Tune kernel parameters for Gaussian process regression.
# Length scale controls smoothness in predictions.
# The plot compares two fixed kernel settings.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

# Create a small noisy one-dimensional training set.
rng = np.random.default_rng(42)
X_train = np.linspace(0, 10, 18).reshape(-1, 1)
y_true = np.sin(X_train).ravel()
y_train = y_true + rng.normal(0, 0.18, size=y_true.shape)

# Validate the basic shape before fitting models.
if X_train.shape[0] != y_train.shape[0]:
    raise ValueError("Training inputs and targets must match.")

# Predict on a dense grid for smooth curves.
X_grid = np.linspace(0, 10, 300).reshape(-1, 1)

# Fit two Gaussian processes with different fixed length scales.
short_kernel = RBF(length_scale=0.35, length_scale_bounds="fixed")
long_kernel = RBF(length_scale=2.0, length_scale_bounds="fixed")

short_gp = GaussianProcessRegressor(kernel=short_kernel, alpha=0.04)
long_gp = GaussianProcessRegressor(kernel=long_kernel, alpha=0.04)

short_gp.fit(X_train, y_train)
long_gp.fit(X_train, y_train)

# Get predictive means and uncertainty bands.
short_mean, short_std = short_gp.predict(X_grid, return_std=True)
long_mean, long_std = long_gp.predict(X_grid, return_std=True)

print("scikit-learn version:", sklearn.__version__)
print("Short length scale: wigglier mean, local uncertainty.")
print("Long length scale: smoother mean, broader assumptions.")

# Plot both tuned kernels on the same training data.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_train.ravel(), y_train, color="black", label="training data")
ax.plot(X_grid.ravel(), short_mean, color="tab:blue", label="length scale 0.35")
ax.plot(X_grid.ravel(), long_mean, color="tab:orange", label="length scale 2.0")

# Show uncertainty for the smoother model only.
ax.fill_between(
    X_grid.ravel(), long_mean - 2 * long_std, long_mean + 2 * long_std,
    color="tab:orange", alpha=0.18, label="long scale uncertainty"
)

ax.set_title("Tuning an RBF Kernel Length Scale")
ax.set_xlabel("Input value")
ax.set_ylabel("Predicted target")
ax.legend()
plt.show()



## **3. GPC Practice**

### **3.1. GP Classification**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_03_01.jpg?v=1784003019" width="250">



>* Predicts class probabilities using latent functions
>* Models nonlinear boundaries with confidence estimates

>* Kernels shape smooth probabilistic class boundaries
>* Classification needs approximate posterior inference

>* Separate predicted labels from model certainty
>* Treat probabilities as calibrated beliefs



In [ ]:
#@title Python Code - GP Classification

# This example fits Gaussian Process Classification.
# It shows probabilities and uncertain boundaries.
# The plot reveals computationally small practice.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_moons
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# A small dataset keeps Gaussian processes practical.
features, labels = make_moons(
    n_samples=160,
    noise=0.25,
    random_state=42,
)

# Check the generated data has the expected shape.
if features.shape != (160, 2):
    raise ValueError("Expected 160 rows and 2 features.")

# Stratification keeps both classes represented in each split.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.30,
    stratify=labels,
    random_state=42,
)

# The RBF kernel says nearby points should behave similarly.
kernel = 1.0 * RBF(length_scale=0.7)
model = GaussianProcessClassifier(
    kernel=kernel,
    random_state=42,
    max_iter_predict=100,
)

# Fitting is intentionally done once on a small dataset.
model.fit(X_train, y_train)

# Probabilities show confidence, not just class labels.
test_probabilities = model.predict_proba(X_test)[:, 1]
test_predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, test_predictions)

print("scikit-learn version:", sklearn.__version__)
print("Test accuracy:", round(accuracy, 3))
print("First five positive-class probabilities:", np.round(test_probabilities[:5], 3))

# A grid lets us visualize the learned probability surface.
x_min = features[:, 0].min() - 0.5
x_max = features[:, 0].max() + 0.5
y_min = features[:, 1].min() - 0.5
y_max = features[:, 1].max() + 0.5

# Keep the grid modest because GPC prediction can be costly.
x_values = np.linspace(x_min, x_max, 90)
y_values = np.linspace(y_min, y_max, 90)
xx, yy = np.meshgrid(x_values, y_values)
grid_points = np.c_[xx.ravel(), yy.ravel()]

# The positive-class probability is highest in yellow regions.
grid_probabilities = model.predict_proba(grid_points)[:, 1]
probability_surface = grid_probabilities.reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
contour = ax.contourf(xx, yy, probability_surface, levels=20, cmap="viridis")
scatter = ax.scatter(
    X_train[:, 0],
    X_train[:, 1],
    c=y_train,
    cmap="coolwarm",
    edgecolor="black",
    s=35,
)

ax.contour(xx, yy, probability_surface, levels=[0.5], colors="white")
ax.set_title("Gaussian Process Classification probabilities")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
fig.colorbar(contour, ax=ax, label="Probability of class 1")
plt.show()



### **3.2. Classification Uncertainty Maps**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_03_02.jpg?v=1784003021" width="250">



>* Maps show predicted classes and confidence
>* Boundaries reveal higher classification uncertainty

>* Separate overlap uncertainty from missing-data uncertainty
>* Use uncertainty to guide decisions and data

>* Kernels and data shape uncertainty patterns
>* Use maps as decision-support summaries



In [ ]:
#@title Python Code - Classification Uncertainty Maps

# This example maps Gaussian process classification uncertainty.
# Colors show predicted probability across feature space.
# Training points reveal where evidence supports confidence.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_moons
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF

# A small two-dimensional dataset keeps GPC fast.
features, labels = make_moons(
    n_samples=80,
    noise=0.25,
    random_state=42,
)

# Validate the shape before fitting the classifier.
if features.shape != (80, 2):
    raise ValueError("Expected 80 rows and 2 features.")

# The RBF kernel creates a smooth probability surface.
kernel = 1.0 * RBF(length_scale=0.6)
model = GaussianProcessClassifier(
    kernel=kernel,
    random_state=42,
    max_iter_predict=100,
)

# Fit once, because Gaussian process classification is computationally costly.
model.fit(features, labels)
training_accuracy = model.score(features, labels)

# Build a small grid for the uncertainty map.
x_min = features[:, 0].min() - 0.6
x_max = features[:, 0].max() + 0.6
y_min = features[:, 1].min() - 0.6
y_max = features[:, 1].max() + 0.6

# Each grid point receives a class-one probability.
x_values = np.linspace(x_min, x_max, 90)
y_values = np.linspace(y_min, y_max, 90)
grid_x, grid_y = np.meshgrid(x_values, y_values)
grid_points = np.c_[grid_x.ravel(), grid_y.ravel()]

# Probabilities near 0.5 indicate high classification uncertainty.
probabilities = model.predict_proba(grid_points)[:, 1]
probability_map = probabilities.reshape(grid_x.shape)
uncertainty = 1.0 - np.abs(probability_map - 0.5) * 2.0

print("scikit-learn version:", sklearn.__version__)
print("Training accuracy:", round(training_accuracy, 3))
print("Mean grid uncertainty:", round(float(uncertainty.mean()), 3))

# One figure shows uncertainty and the observed training labels.
fig, ax = plt.subplots(figsize=(7, 5))
image = ax.contourf(
    grid_x,
    grid_y,
    uncertainty,
    levels=20,
    cmap="viridis",
)

# Training points show where the model has direct evidence.
scatter = ax.scatter(
    features[:, 0],
    features[:, 1],
    c=labels,
    cmap="coolwarm",
    edgecolor="black",
    s=45,
)

ax.contour(
    grid_x,
    grid_y,
    probability_map,
    levels=[0.5],
    colors="white",
    linewidths=2,
)

ax.set_title("GPC classification uncertainty map")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
fig.colorbar(image, ax=ax, label="Uncertainty, highest near 0.5")
plt.show()



### **3.3. Scaling Limits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_03_03.jpg?v=1784003023" width="250">



>* Predicts classes with useful confidence estimates
>* Becomes costly as training data grows

>* Large datasets slow training and prediction
>* Match GPC to size and speed needs

>* Match GPC to data size and uncertainty
>* Balance accuracy, cost, and practical deployment



# <font color="#418FDE" size="6.5" uppercase>**Gaussian Processes**</font>


In this lecture, you learned to:
- Fit Gaussian Process Regression and interpret predictive mean and uncertainty. 
- Configure and compose Gaussian process kernels for practical modeling. 
- Apply Gaussian Process Classification while recognizing computational limits. 

In the next Lecture (Lecture B), we will go over 'Cross Decomposition'